In [2]:
import tensorflow_hub as hub
import tensorflow_text as text


In [3]:
encoder_url="https://tfhub.dev/tensorflow/small_bert/bert_en_uncased_L-4_H-512_A-8/1"
preprocess_url="https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3"

In [4]:
bert_preprocess_model = hub.KerasLayer(preprocess_url)
bert_encoder_model = hub.KerasLayer(encoder_url)

In [6]:
text_test=['This is good','This is bad']

text_preprocessed=bert_preprocess_model(text_test)
print(text_preprocessed)

{'input_word_ids': <tf.Tensor: shape=(2, 128), dtype=int32, numpy=
array([[ 101, 2023, 2003, 2204,  102,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0],
       [ 101, 2023, 2003, 2919,  102,    0,    0,    0,    0,    0, 

In [7]:
text_preprocessed.keys()

dict_keys(['input_word_ids', 'input_type_ids', 'input_mask'])

In [8]:
bert_result=bert_encoder_model(text_preprocessed)

In [9]:
bert_result.keys()

dict_keys(['pooled_output', 'default', 'sequence_output', 'encoder_outputs'])

In [10]:
bert_result['pooled_output']

<tf.Tensor: shape=(2, 512), dtype=float32, numpy=
array([[ 0.9919021 ,  0.6373504 ,  0.07190085, ...,  0.09786866,
        -0.57325226, -0.3467424 ],
       [ 0.9922416 ,  0.4221559 ,  0.05266318, ...,  0.21491493,
        -0.2602834 , -0.00973877]], dtype=float32)>

In [11]:
bert_result['sequence_output']

<tf.Tensor: shape=(2, 128, 512), dtype=float32, numpy=
array([[[ 0.86761796,  1.118505  , -0.09938756, ..., -0.58379257,
          0.05104278, -0.09694657],
        [ 0.17673856,  0.34169966, -0.27889273, ...,  0.37430316,
          0.10265742,  0.7348001 ],
        [ 0.1198582 ,  1.1314502 , -1.2027926 , ..., -0.11834525,
          0.21292815,  0.3188154 ],
        ...,
        [ 0.48468918,  0.5784715 , -0.6832554 , ..., -0.22296363,
          0.3884541 ,  0.45885298],
        [ 1.0812602 ,  0.38242936, -0.9161845 , ..., -0.26350212,
          0.18948406, -0.04497535],
        [ 1.0510592 ,  0.84697145, -0.8383443 , ..., -0.09361617,
          0.53514624, -0.45647815]],

       [[ 0.6923341 ,  0.89509124, -0.01082072, ..., -0.27644017,
         -0.2739247 , -0.5552534 ],
        [ 0.05526368,  0.32337737, -0.3168517 , ...,  0.10192323,
         -0.132247  ,  0.2787647 ],
        [ 0.04811946,  0.9522676 , -1.1899763 , ..., -0.00519184,
         -0.10925719, -0.21804565],
        ...,

In [12]:
len(bert_result['encoder_outputs'])

4

**Email Classification**

In [18]:
import pandas as pd

In [19]:
df=pd.read_csv("./dataset/spam.csv")

In [20]:
df.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [21]:
df['Category'].value_counts()

Category
ham     4825
spam     747
Name: count, dtype: int64

In [22]:
747/4825

0.15481865284974095

In [24]:
df_ham=df[df['Category']=='ham']
df_spam=df[df['Category']=='spam']

In [25]:
df_downsample=pd.concat([df_ham.sample(747),df_spam],axis=0)    

In [26]:
df_downsample['Category'].value_counts()

Category
ham     747
spam    747
Name: count, dtype: int64

In [27]:
df_downsample['spam']=df_downsample['Category'].apply(lambda x: 1 if x=='spam' else 0)

In [28]:
df_downsample.head()

,Category,Message,spam
2294,ham,Hello. Damn this christmas thing. I think i ha...,0
1496,ham,Hey gals.. Anyone of u going down to e driving...,0
4748,ham,"When you just put in the + sign, choose my num...",0
454,ham,Ok i will tell her to stay out. Yeah its been ...,0
996,ham,Change again... It's e one next to escalator...,0


In [65]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df_downsample['Message'],df_downsample['spam'],test_size=0.2,random_state=42,stratify=df_downsample['spam'])

In [66]:
X_train = X_train.reset_index(drop=True)
X_test  = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test  = y_test.reset_index(drop=True)


In [67]:
X_train.head(), y_train.head()


(0    You are a winner U have been specially selecte...
 1    Welcome to Select, an O2 service with added be...
 2                  Lmao. Take a pic and send it to me.
 3    FreeMsg: Hey - I'm Buffy. 25 and love to satis...
 4    Orange brings you ringtones from all time Char...
 Name: Message, dtype: object,
 0    1
 1    1
 2    0
 3    1
 4    1
 Name: spam, dtype: int64)

In [68]:
def prepare_bert_input(texts):
    text_preprocessed=bert_preprocess_model(texts)
    return bert_encoder_model(text_preprocessed)['pooled_output']

In [69]:
e=prepare_bert_input(X_train[:3])

In [70]:
from sklearn.metrics.pairwise import cosine_similarity

In [71]:
cosine_similarity([e[0]],[e[2]])

array([[0.59485793]], dtype=float32)

In [72]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models

In [73]:
#Bert Layer

text_input = layers.Input(shape=(), dtype=tf.string, name='text')
process_text=bert_preprocess_model(text_input)
outputs=bert_encoder_model(process_text)

#Neural Network Layers
l=layers.Dropout(0.1,name="dropout")(outputs['pooled_output'])
l=layers.Dense(1,activation='sigmoid',name="output")(l)


#Construct final Model

model=keras.Model(inputs=[text_input],outputs=[l])




In [74]:
model.summary()


Model: "model_2"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 text (InputLayer)              [(None,)]            0           []                               
                                                                                                  
 keras_layer (KerasLayer)       {'input_word_ids':   0           ['text[0][0]']                   
                                (None, 128),                                                      
                                 'input_type_ids':                                                
                                (None, 128),                                                      
                                 'input_mask': (Non                                               
                                e, 128)}                                                    

In [75]:
METRICS=[
    keras.metrics.BinaryAccuracy(name='accuracy'),
    keras.metrics.Precision(name='precision'),
    keras.metrics.Recall(name='recall'),
]

model.compile(optimizer='adam',loss='binary_crossentropy',metrics=METRICS)

In [76]:
model.fit(X_train,y_train,epochs=10)

Epoch 1/10
38/38 [==============================] - 77s 2s/step - loss: 0.7235 - accuracy: 0.5314 - precision: 0.5342 - recall: 0.4967
Epoch 2/10
38/38 [==============================] - 75s 2s/step - loss: 0.4284 - accuracy: 0.8695 - precision: 0.8542 - recall: 0.8913
Epoch 3/10
38/38 [==============================] - 77s 2s/step - loss: 0.3082 - accuracy: 0.9222 - precision: 0.9229 - recall: 0.9214
Epoch 4/10
38/38 [==============================] - 69s 2s/step - loss: 0.2477 - accuracy: 0.9389 - precision: 0.9282 - recall: 0.9515
Epoch 5/10
38/38 [==============================] - 72s 2s/step - loss: 0.2179 - accuracy: 0.9406 - precision: 0.9271 - recall: 0.9565
Epoch 6/10
38/38 [==============================] - 80s 2s/step - loss: 0.1937 - accuracy: 0.9473 - precision: 0.9436 - recall: 0.9515
Epoch 7/10
38/38 [==============================] - 77s 2s/step - loss: 0.1754 - accuracy: 0.9556 - precision: 0.9460 - recall: 0.9666
Epoch 8/10
38/38 [==============================] - 81s

In [80]:
model.save('bert_spam_detector_model.model.h5')

In [79]:
model.evaluate(X_test,y_test)

10/10 [==============================] - 30s 3s/step - loss: 0.1829 - accuracy: 0.9498 - precision: 0.9589 - recall: 0.9396


[0.18291820585727692,
 0.9498327970504761,
 0.9589040875434875,
 0.9395973086357117]